In [ ]:
import requests
import os

#This downloads the monthy airline ontime perfomance from the Beureau of Trransporation Statistics 

def download_bts_months(year_month_pairs, output_dir="bts_data"): # This function downloads the specified year-month pairs from the BTS website and saves them to the output directory.
    os.makedirs(output_dir, exist_ok=True)
    base_url = ("https://transtats.bts.gov/PREZIP/" "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{}_{}.zip")
    for year, month in year_month_pairs: # Loops years and months 
        url = base_url.format(year, month)
        filename = f"ontime_{year}_{month:02d}.zip" 
        filepath = os.path.join(output_dir, filename) 
        print(f"Downloading {year}-{month}")
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(filepath, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192): # Avoids memory issues 
                    f.write(chunk)
            print(f"Saved {filename}")
        else:
            print(f"  FAILED {year}-{month} (status {response.status_code})")
# April 2024 through December 2025 (12 months)
months = ([(2024, m) for m in range(4, 13)] + [(2025, m) for m in range(1, 13)])

download_bts_months(months)

  Saved ontime_2024_04.zip
  Saved ontime_2024_05.zip
  Saved ontime_2024_06.zip
  Saved ontime_2024_07.zip
  Saved ontime_2024_08.zip
  Saved ontime_2024_09.zip
  Saved ontime_2024_10.zip
  Saved ontime_2024_11.zip
  Saved ontime_2024_12.zip
  Saved ontime_2025_01.zip
  Saved ontime_2025_02.zip
  Saved ontime_2025_03.zip
  Saved ontime_2025_04.zip
  Saved ontime_2025_05.zip
  Saved ontime_2025_06.zip
  Saved ontime_2025_07.zip
  Saved ontime_2025_08.zip
  Saved ontime_2025_09.zip
  Saved ontime_2025_10.zip
  Saved ontime_2025_11.zip
  Saved ontime_2025_12.zip
  Saved ontime_2026_01.zip
  Saved ontime_2026_02.zip


In [ ]:
import zipfile
import pandas as pd

# Defines the cloumns that are relievant for my analysis 

COLUMNS_TO_KEEP = ["FlightDate", "Reporting_Airline", "Tail_Number",
    "Flight_Number_Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "DepDelay","DepDelayMinutes", "DepDel15",
    "ArrDelay", "ArrDelayMinutes", "ArrDel15",
    "Cancelled","CancellationCode", "Diverted",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay","Distance"
] 

def load_all_months(data_dir="bts_data"): # Load and extract the specified columns 
    frames = []
    for filename in sorted(os.listdir(data_dir)):
        if filename.endswith(".zip"):
            with zipfile.ZipFile(os.path.join(data_dir, filename)) as z:
                csv_name =[n for n in z.namelist() if n.endswith(".csv")][0]
                with z.open(csv_name) as f: 
                    df = pd.read_csv(f, usecols=COLUMNS_TO_KEEP, low_memory=False)
                    frames.append(df)
                    print(f"Loaded {filename}: {len(df):,} rows")
    combined = pd.concat(frames, ignore_index=True)
    return combined

full_data = load_all_months()
print(f"\nTotal rows: {len(full_data):,}")
print(f"Memory usage: {full_data.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Date range: {full_data.FlightDate.min()} to {full_data.FlightDate.max()}")

Loaded ontime_2024_04.zip: 582,185 rows
Loaded ontime_2024_05.zip: 609,743 rows
Loaded ontime_2024_06.zip: 611,132 rows
Loaded ontime_2024_07.zip: 634,613 rows
Loaded ontime_2024_08.zip: 619,025 rows
Loaded ontime_2024_09.zip: 582,622 rows
Loaded ontime_2024_10.zip: 615,497 rows
Loaded ontime_2024_11.zip: 575,404 rows
Loaded ontime_2024_12.zip: 590,581 rows
Loaded ontime_2025_01.zip: 539,747 rows
Loaded ontime_2025_02.zip: 504,884 rows
Loaded ontime_2025_03.zip: 600,872 rows
Loaded ontime_2025_04.zip: 583,950 rows
Loaded ontime_2025_05.zip: 605,648 rows
Loaded ontime_2025_06.zip: 611,575 rows
Loaded ontime_2025_07.zip: 631,428 rows
Loaded ontime_2025_08.zip: 602,378 rows
Loaded ontime_2025_09.zip: 562,439 rows
Loaded ontime_2025_10.zip: 605,844 rows
Loaded ontime_2025_11.zip: 570,550 rows
Loaded ontime_2025_12.zip: 582,304 rows

Total rows: 12,422,421
Memory usage: 2487.4 MB
Date range: 2024-04-01 to 2025-12-31


In [ ]:
full_data["FlightDate"] = pd.to_datetime(full_data["FlightDate"]) # Flight data converted to datetime 
full_data["CRSDepTime"] = full_data["CRSDepTime"].astype("Int64") # Converst to ints 

for col in ["Reporting_Airline", "Origin", "Dest", "CancellationCode", "Tail_Number"]:
    full_data[col] = full_data[col].astype("category") # Saves memory 

print(full_data.dtypes)
print(f"\nMemory after type fixes: {full_data.memory_usage(deep=True).sum() / 1e6:.1f} MB")

FlightDate                         datetime64[us]
Reporting_Airline                        category
Tail_Number                              category
Flight_Number_Reporting_Airline           float64
Origin                                   category
Dest                                     category
CRSDepTime                                  Int64
DepDelay                                  float64
DepDelayMinutes                           float64
DepDel15                                  float64
ArrDelay                                  float64
ArrDelayMinutes                           float64
ArrDel15                                  float64
Cancelled                                 float64
CancellationCode                         category
Diverted                                  float64
Distance                                  float64
CarrierDelay                              float64
WeatherDelay                              float64
NASDelay                                  float64


In [ ]:
#Unique values in key columns
print(f"Unique origins: {full_data['Origin'].nunique()}")
print(f"Unique dests: {full_data['Dest'].nunique()}")
print(f"Unique airlines: {full_data['Reporting_Airline'].nunique()}")
print(f"Unique tails: {full_data['Tail_Number'].nunique()}")

# What fraction of flights are delayed, cancelled, or diverted?
print(f"\nDelayed (15+ min): {full_data['DepDel15'].mean():.1%}")
print(f"Cancelled: {full_data['Cancelled'].mean():.1%}")
print(f"Diverted: {full_data['Diverted'].mean():.1%}")

# Any months missing?
print(f"\nFlights per month:")
print(full_data.groupby(full_data["FlightDate"].dt.to_period("M")).size().to_string())

Unique origins:   356
Unique dests:     356
Unique airlines:  15
Unique tails:     6424

Delayed (15+ min): 21.4%
Cancelled:         1.4%
Diverted:          0.3%

Flights per month:
FlightDate
2024-04    582185
2024-05    609743
2024-06    611132
2024-07    634613
2024-08    619025
2024-09    582622
2024-10    615497
2024-11    575404
2024-12    590581
2025-01    539747
2025-02    504884
2025-03    600872
2025-04    583950
2025-05    605648
2025-06    611575
2025-07    631428
2025-08    602378
2025-09    562439
2025-10    605844
2025-11    570550
2025-12    582304
Freq: M


In [ ]:
#Parquet is supports data compression
full_data.to_parquet("bts_combined.parquet", index=False)
print("Saved to bts_combined.parquet")

Saved to bts_combined.parquet
